In [ ]:
!pip install gradio openai

In [ ]:
import gradio as gr
from openai import OpenAI

API_KEY = API_KEY = "PASTE_YOUR_FREE_GROQ_KEY_HERE"   # get one free at https://console.groq.com/keys

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=API_KEY)
MODEL = "openai/gpt-oss-20b"

def llm(system, user):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        temperature=0.7,
    )
    return resp.choices[0].message.content.strip()

def generate_question(role):
    role = (role or "Software Engineer").strip()
    system = ("You are a senior technical interviewer. Ask ONE concise interview "
              "question for a campus / early-career candidate. Output only the question.")
    try:
        q = llm(system, f"Role: {role}")
        return q, q
    except Exception as e:
        return f"⚠️ {e}", ""

def score_answer(role, question, answer):
    if not question:
        return "Click **Generate question** first."
    if not (answer or "").strip():
        return "Type an answer before scoring."
    system = ("You are a rigorous but fair interview coach. Score the candidate's answer.\n"
              "Reply in EXACTLY this format:\n"
              "**Score:** X/10\n**What worked:** <1-2 lines>\n"
              "**What was missing:** <1-2 lines>\n**Stronger answer:** <2-3 lines>")
    try:
        return llm(system, f"Role: {role}\nQuestion: {question}\nCandidate answer: {answer}")
    except Exception as e:
        return f"⚠️ {e}"

with gr.Blocks(title="PrepPilot") as demo:
    gr.Markdown("# 🚀 PrepPilot — AI Mock Interview Coach")
    with gr.Row():
        role = gr.Textbox(label="Role", value="Software Engineer", scale=3)
        gen_btn = gr.Button("Generate question", variant="primary", scale=1)
    question_box = gr.Textbox(label="Interview question", interactive=False, lines=2)
    current_q = gr.State("")
    answer = gr.Textbox(label="Your answer", lines=6)
    score_btn = gr.Button("Score my answer", variant="primary")
    feedback = gr.Markdown("*Your feedback will appear here.*")
    gen_btn.click(generate_question, inputs=role, outputs=[question_box, current_q])
    score_btn.click(score_answer, inputs=[role, current_q, answer], outputs=feedback)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2de24d3a09cb90a629.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

TOTAL_QUESTIONS = 5

def make_question(role, asked):
    asked_txt = "\n".join(f"- {q}" for q in asked) if asked else "(none yet)"
    system = ("You are a senior interviewer running a mock interview. Ask ONE new interview "
              "question for the role. Vary between technical and behavioral. Do NOT repeat any "
              "already-asked question. Output only the question.")
    return llm(system, f"Role: {role}\nAlready asked:\n{asked_txt}")

def score_one(role, question, answer):
    system = ("You are a fair interview coach. Score this single answer.\n"
              "Reply EXACTLY as:\n"
              "**Score:** X/10 — **What worked:** <1 line> — **Missing:** <1 line>")
    return llm(system, f"Role: {role}\nQ: {question}\nAnswer: {answer}")

def build_report(session):
    lines = [f"Q{i}: {t['q']}\nAnswer: {t['a']}\nNote: {t['feedback']}"
             for i, t in enumerate(session["transcript"], 1)]
    system = ("You are an interview coach writing a final report. Based on the whole interview, give:\n"
              "**Overall:** X/10\n**Top strengths:** 2-3 bullets\n"
              "**Work on this:** 2-3 bullets\n**One thing to do next:** 1 line")
    return llm(system, f"Role: {session['role']}\n\nInterview:\n" + "\n\n".join(lines))

def q_display(q, idx):
    return f"### Question {idx} of {TOTAL_QUESTIONS}\n{q}"

def start_interview(role):
    role = (role or "Software Engineer").strip()
    try:
        q1 = make_question(role, [])
    except Exception as e:
        return f"⚠️ {e}", "", "", None
    session = {"role": role, "idx": 1, "current_q": q1, "transcript": []}
    return q_display(q1, 1), "", "", session

def submit_answer(answer, session):
    if not session:
        return "Click **Start interview** first.", "", "", session
    if not (answer or "").strip():
        return q_display(session["current_q"], session["idx"]), answer, "", session
    try:
        fb = score_one(session["role"], session["current_q"], answer)
    except Exception as e:
        return q_display(session["current_q"], session["idx"]), answer, f"⚠️ {e}", session
    session["transcript"].append({"q": session["current_q"], "a": answer, "feedback": fb})
    if session["idx"] < TOTAL_QUESTIONS:
        asked = [t["q"] for t in session["transcript"]]
        nxt = make_question(session["role"], asked)
        session["idx"] += 1
        session["current_q"] = nxt
        return q_display(nxt, session["idx"]), "", f"*Last answer* — {fb}", session
    report = build_report(session)
    return "### ✅ Interview complete — see your report below.", "", report, session

with gr.Blocks(title="PrepPilot") as demo:
    gr.Markdown("# 🚀 PrepPilot — AI Mock Interview Coach")
    gr.Markdown("A full 5-question mock interview with live feedback and a final report.")
    with gr.Row():
        role = gr.Textbox(label="Role", value="Software Engineer", scale=3)
        start_btn = gr.Button("Start interview", variant="primary", scale=1)
    question_box = gr.Markdown("*Click Start interview to begin.*")
    session = gr.State(None)
    answer = gr.Textbox(label="Your answer", lines=6, placeholder="Type your answer, then click Submit...")
    submit_btn = gr.Button("Submit answer", variant="primary")
    report_box = gr.Markdown("")
    start_btn.click(start_interview, inputs=role, outputs=[question_box, answer, report_box, session])
    submit_btn.click(submit_answer, inputs=[answer, session], outputs=[question_box, answer, report_box, session])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://38e17f4219a520cb16.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q sentence-transformers

import json, re
import numpy as np
from sentence_transformers import CrossEncoder

# your NLI cross-encoder — the scoring engine from your ACL / answer-sheet work
nli = CrossEncoder("cross-encoder/nli-deberta-v3-base")   # labels: [contradiction, entailment, neutral]

def softmax(x):
    e = np.exp(x - np.max(x)); return e / e.sum()

def generate_key_points(role, question):
    """LLM turns the ideal answer into 3-5 atomic 'Golden Key' points (the hypotheses)."""
    system = ("You are an expert interviewer. List the 3-5 essential points an ideal answer to the "
              "question MUST cover. Return ONLY a JSON array of short strings. No other text.")
    raw = llm(system, f"Role: {role}\nQuestion: {question}")
    txt = raw.strip().strip("`").strip()
    if txt.lower().startswith("json"):
        txt = txt[4:].strip()
    try:
        pts = json.loads(txt)
        if isinstance(pts, list) and pts:
            return [str(p) for p in pts][:5]
    except Exception:
        pass
    lines = [re.sub(r'^[\s\-\*\d\.\)]+', '', l).strip() for l in raw.splitlines() if l.strip()]
    return [l for l in lines if l][:5] or ["A clear, correct, relevant answer"]

def nli_score(answer, key_points):
    """Cross-encoder checks whether the answer ENTAILS each key point — your paper's method."""
    logits = np.array(nli.predict([(answer, kp) for kp in key_points]))
    results = [(kp, float(softmax(row)[1])) for kp, row in zip(key_points, logits)]  # [1] = entailment
    score = round(sum(e for _, e in results) / len(results) * 10, 1)
    covered = [f"✓ {kp}  ({e:.2f})" for kp, e in results if e >= 0.5]
    missed  = [f"✗ {kp}  ({e:.2f})" for kp, e in results if e < 0.5]
    lines = [f"**Score:** {score}/10  _(NLI cross-encoder vs {len(results)} key points)_"]
    if covered: lines.append("**Covered:**\n" + "\n".join(covered))
    if missed:  lines.append("**Missed:**\n" + "\n".join(missed))
    return "\n\n".join(lines)

# drop-in replacement for the old LLM-only scorer — same name, so your interview app just picks it up
def score_one(role, question, answer):
    try:
        return nli_score(answer, generate_key_points(role, question))
    except Exception as e:
        return f"⚠️ {e}"

print("✅ Cross-encoder scorer ready.")

KeyboardInterrupt: 